In [ ]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

In [ ]:
zones = pd.read_csv('../data/processed/pjm_weather_zones.csv')
zones

In [ ]:
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [ ]:
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
hourly_vars = ["temperature_2m", "precipitation", "wind_speed_10m", "cloud_cover"]

zone_frames = []
for _, row in zones.iterrows():
    params = {
        "latitude": row["lat"],
        "longitude": row["lon"],
        "start_date": "2023-01-01",
        "end_date": "2026-08-07",
        "hourly": hourly_vars,
        "timezone": "UTC",
    }
    response = openmeteo.weather_api(url, params=params)[0]
    hourly = response.Hourly()

    df = pd.DataFrame({
        "time": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        ),
        "temperature": hourly.Variables(0).ValuesAsNumpy(),
        "precipitation": hourly.Variables(1).ValuesAsNumpy(),
        "wind_speed": hourly.Variables(2).ValuesAsNumpy(),
        "cloud_cover": hourly.Variables(3).ValuesAsNumpy(),
    })
    df["zone"] = row["zone_id"]
    zone_frames.append(df)

In [ ]:
weather_hourly = pd.concat(zone_frames, ignore_index=True)
weather_hourly = weather_hourly[["time", "zone", "temperature", "precipitation", "wind_speed", "cloud_cover"]]
weather_hourly

In [ ]:
weather_hourly.to_csv("../data/external/openmeteo_hourly_2023_present.csv", index=False)